# 09 — Hybrid: full-context linear-attn memory + short vanilla DT

A two-stream architecture for the cross-trial-memory problem:

- **Long stream** (`LongLinearMemory`) — causal linear-attention over the *entire* session. Emits a per-step memory vector at every state position. Cheap because linear attention is `O(L)`.
- **Short stream** (`DecisionTransformer`) — vanilla softmax DT with a small context (32-256 steps). The action head.

The long-stream hidden at step `t` is **added to the short DT's state input at step `t`** ("suggestions transposed onto the main DT"). The short DT then attends causally over the short window as usual; the additive bias lets it condition on cross-trial structure the short window can't see directly.

Trained end-to-end: each step picks a small batch of sessions, encodes the full session through the long stream once, samples a handful of short windows per session, sums long-stream memory into the short window's state inputs, and backprops a single cross-entropy loss through both streams.


## 1. Setup


In [ ]:
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

import sys; sys.path.insert(0, '../scripts')
import train_dt as T

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.decision_transformer import DTConfig, DecisionTransformer
from corner_maze_rl.models.linear_decision_transformer import LinearAttnEncoderLayer

DATA_PATH    = Path('../data/yoked/dataset/actions_synthetic_pretrial.parquet')
RUN_DIR      = Path('../runs/dt/nb09'); RUN_DIR.mkdir(parents=True, exist_ok=True)

FULL_CTX     = 4096     # long-stream cap (~99th percentile of session length)
SHORT_CTX    = 64       # short-stream context
EMBED_DIM    = 60
NUM_HEADS    = 4
LONG_LAYERS  = 2
SHORT_LAYERS = 2
LR           = 5e-4
WEIGHT_DECAY = 1e-4
SESSIONS_PER_STEP   = 4   # long-stream batch
WINDOWS_PER_SESSION = 8   # short-stream samples per session per step
EPOCHS       = 30
VAL_FRAC     = 0.10
SEED         = 0
MAX_SESSIONS = None

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={device}  FULL_CTX={FULL_CTX}  SHORT_CTX={SHORT_CTX}')


## 2. Hybrid model


In [ ]:
class LongLinearMemory(nn.Module):
    """Causal linear-attn over the full session.

    Emits per-step hidden vectors at state-token positions, shape (B, L, D).
    Same I/O contract as LinearDecisionTransformer minus the action head.
    """
    def __init__(self, embed_dim, num_heads, num_layers,
                 num_actions=5, full_ctx=FULL_CTX,
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.embed_rtg    = nn.Linear(1, embed_dim)
        self.embed_state  = nn.Linear(embed_dim, embed_dim)
        self.embed_action = nn.Linear(num_actions, embed_dim)
        self.position_emb = nn.Embedding(full_ctx, embed_dim)
        self.layers = nn.ModuleList([
            LinearAttnEncoderLayer(embed_dim, num_heads, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, rtg, state, action):
        b, k, _ = state.shape
        r = self.embed_rtg(rtg); s = self.embed_state(state); a = self.embed_action(action)
        p = self.position_emb(torch.arange(k, device=rtg.device)).unsqueeze(0)
        r, s, a = r + p, s + p, a + p
        x = torch.stack((r, s, a), dim=2).reshape(b, 3 * k, self.embed_dim)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return x[:, 1::3, :]   # state-position hiddens: (B, L, D)


class HybridDT(nn.Module):
    """Long-stream memory + short-stream vanilla DT.

    The long stream emits a memory vector at every timestep; the short DT
    receives the rat's pose state PLUS the long-stream memory at that step,
    added before the short DT's own state embedding.
    """
    def __init__(self, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 long_layers=LONG_LAYERS, short_layers=SHORT_LAYERS,
                 full_ctx=FULL_CTX, short_ctx=SHORT_CTX, num_actions=5):
        super().__init__()
        self.long = LongLinearMemory(embed_dim, num_heads, long_layers,
                                     num_actions=num_actions, full_ctx=full_ctx)
        short_cfg = DTConfig(embed_dim=embed_dim, num_actions=num_actions,
                             context_size=short_ctx, num_heads=num_heads,
                             num_layers=short_layers, pos_encoding='learned')
        self.short = DecisionTransformer(short_cfg)
        self.embed_dim = embed_dim
        self.short_ctx = short_ctx

    def long_encode(self, rtg_full, state_full, action_full):
        return self.long(rtg_full, state_full, action_full)

    def short_forward(self, rtg_win, state_win, action_win, mem_win):
        return self.short(rtg_win, state_win + mem_win, action_win)


## 3. Load + pack each session


In [ ]:
df = pd.read_parquet(DATA_PATH)
if MAX_SESSIONS is not None:
    keep = sorted(df['session_id'].unique())[:MAX_SESSIONS]
    df = df[df['session_id'].isin(keep)].reset_index(drop=True)

encoder = GridCellEncoder()
assert encoder.output_dim == EMBED_DIM

def pack_full(sdf, full_ctx):
    arrays = T.encode_session(sdf, encoder)
    n = arrays['state'].shape[0]
    if n > full_ctx:
        arrays = {k: v[-full_ctx:] for k, v in arrays.items()}
        n = full_ctx
    state  = np.zeros((full_ctx, EMBED_DIM), dtype=np.float32); state[:n]  = arrays['state']
    action = np.zeros((full_ctx, 5),         dtype=np.float32); action[:n] = arrays['action']
    rtg    = np.zeros((full_ctx, 1),         dtype=np.float32); rtg[:n]    = arrays['rtg']
    return state, action, rtg, n

sids = df['session_id'].unique().tolist()
sessions = []
for sid in sids:
    sdf = df[df['session_id'] == sid].sort_values('step')
    s, a, r, n = pack_full(sdf, FULL_CTX)
    sessions.append({'sid': sid, 'state': s, 'action': a, 'rtg': r, 'n': n})

rng = np.random.default_rng(SEED)
order = rng.permutation(len(sessions))
n_val = max(1, int(len(sessions) * VAL_FRAC))
val_idx   = order[:n_val].tolist()
train_idx = order[n_val:].tolist()
print(f'sessions: train={len(train_idx)}  val={len(val_idx)}   FULL_CTX={FULL_CTX}')


## 4. Train (joint, end-to-end)


In [ ]:
model = HybridDT().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce = nn.CrossEntropyLoss()
n_params = sum(p.numel() for p in model.parameters())
print(f'params={n_params:,}')

def stack_batch(indices):
    rtg    = torch.from_numpy(np.stack([sessions[i]['rtg']    for i in indices])).to(device)
    state  = torch.from_numpy(np.stack([sessions[i]['state']  for i in indices])).to(device)
    action = torch.from_numpy(np.stack([sessions[i]['action'] for i in indices])).to(device)
    ns     = np.array([sessions[i]['n'] for i in indices], dtype=np.int64)
    return rtg, state, action, ns

def sample_windows(ns, m, rng):
    """For each session length n_i, sample m random window-end positions in [SHORT_CTX-1, n_i)."""
    out = np.empty((len(ns), m), dtype=np.int64)
    for i, n in enumerate(ns):
        hi = max(SHORT_CTX, int(n))
        out[i] = rng.integers(low=SHORT_CTX - 1, high=hi, size=m)
    return torch.from_numpy(out)

def gather_windows(full, end_idx):
    """full: (B, L_full, D)   end_idx: (B, M)   -> (B*M, SHORT_CTX, D)."""
    B, L, D = full.shape
    M = end_idx.shape[1]
    offs = torch.arange(SHORT_CTX, device=full.device).view(1, 1, SHORT_CTX)
    starts = (end_idx.to(full.device) - (SHORT_CTX - 1)).unsqueeze(-1)
    idx = (starts + offs).clamp(min=0)
    idx_exp = idx.unsqueeze(-1).expand(B, M, SHORT_CTX, D)
    full_exp = full.unsqueeze(1).expand(B, M, L, D)
    return torch.gather(full_exp, dim=2, index=idx_exp).reshape(B * M, SHORT_CTX, D)

def run_epoch(idx_list, train: bool, rng):
    model.train() if train else model.eval()
    sum_loss = sum_correct = sum_tokens = 0.0
    ctx_mgr = torch.enable_grad() if train else torch.no_grad()
    idx_arr = np.array(idx_list)
    if train: rng.shuffle(idx_arr)
    steps = len(idx_arr) // SESSIONS_PER_STEP
    with ctx_mgr:
        for st in range(steps):
            batch_idx = idx_arr[st * SESSIONS_PER_STEP:(st + 1) * SESSIONS_PER_STEP].tolist()
            rtg, state, action, ns = stack_batch(batch_idx)
            mem = model.long_encode(rtg, state, action)
            ends = sample_windows(ns, WINDOWS_PER_SESSION, rng)
            rtg_w    = gather_windows(rtg,    ends)
            state_w  = gather_windows(state,  ends)
            action_w = gather_windows(action, ends)
            mem_w    = gather_windows(mem,    ends)
            logits = model.short_forward(rtg_w, state_w, action_w, mem_w)
            targets = action_w.argmax(dim=-1)
            loss = ce(logits.reshape(-1, 5), targets.reshape(-1))
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            sum_loss    += loss.item() * targets.numel()
            sum_correct += (logits.argmax(-1) == targets).sum().item()
            sum_tokens  += targets.numel()
    return sum_loss / max(sum_tokens, 1), sum_correct / max(sum_tokens, 1)

history = []
t0 = time.time()
rng = np.random.default_rng(SEED)
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_idx, train=True,  rng=rng)
    val_loss,   val_acc   = run_epoch(val_idx,   train=False, rng=rng)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    print(f'[epoch {epoch:3d}] train_loss={train_loss:.4f} acc={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} acc={val_acc:.3f}  ({time.time()-t0:.0f}s)')

ckpt = RUN_DIR / 'model.pt'
torch.save({'state_dict': model.state_dict(), 'arch': 'hybrid',
            'short_ctx': SHORT_CTX, 'full_ctx': FULL_CTX,
            'embed_dim': EMBED_DIM, 'num_heads': NUM_HEADS,
            'long_layers': LONG_LAYERS, 'short_layers': SHORT_LAYERS}, ckpt)
(RUN_DIR / 'metrics.jsonl').write_text(''.join(json.dumps(h)+'\n' for h in history))
print(f'\nsaved checkpoint -> {ckpt}')


## 5. Inference movie

Re-encode the growing rollout history through the long stream each step (`O(L)` linear attention; cheap), gather long-stream memory at the latest `SHORT_CTX` positions, sum it into the short DT's state inputs.


In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw

RTG_TARGET   = 20.0
MAX_STEPS    = 600
BASE_TEMP    = 1.0
ROLLOUT_SEED = 0
MOVIE_PATH   = RUN_DIR / 'rollout.mp4'
ACTION_NAMES = ['Left', 'Right', 'Forward', 'EnterWell', 'Pause']

model.eval()
env = CornerMazeEnv(session_type='exposure', obs_mode='view', render_mode='rgb_array')
env.reset(seed=ROLLOUT_SEED)

hist_state, hist_action, hist_rtg = [], [], []
frames = []
last_pose = None; stagnation = 0
total_reward = 0.0; n_well_rewards = 0

for step in range(MAX_STEPS):
    x, y, d = int(env.agent_pos[0]), int(env.agent_pos[1]), int(env.agent_dir)
    if (x, y, d) == last_pose: stagnation += 1
    else: stagnation = 0
    last_pose = (x, y, d)
    temp = BASE_TEMP + (stagnation // 3) * 1.5

    s_vec = encoder.encode(x, y, d).astype(np.float32)
    hist_state.append(s_vec)
    hist_rtg.append(np.array([RTG_TARGET], dtype=np.float32))
    if len(hist_action) < len(hist_state):
        hist_action.append(np.zeros(5, dtype=np.float32))

    L = len(hist_state)
    if L > FULL_CTX:
        hist_state  = hist_state[-FULL_CTX:]
        hist_action = hist_action[-FULL_CTX:]
        hist_rtg    = hist_rtg[-FULL_CTX:]
        L = FULL_CTX
    rtg_t    = torch.from_numpy(np.stack(hist_rtg)).unsqueeze(0).to(device)
    state_t  = torch.from_numpy(np.stack(hist_state)).unsqueeze(0).to(device)
    action_t = torch.from_numpy(np.stack(hist_action)).unsqueeze(0).to(device)

    with torch.no_grad():
        mem_full = model.long_encode(rtg_t, state_t, action_t)   # (1, L, D)
        start = max(0, L - SHORT_CTX)
        rtg_w    = rtg_t   [:, start:L]
        state_w  = state_t [:, start:L]
        action_w = action_t[:, start:L]
        mem_w    = mem_full[:, start:L]
        logits = model.short_forward(rtg_w, state_w, action_w, mem_w)
        probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)
        action = int(torch.multinomial(probs, num_samples=1).item())

    img = env.render()
    canvas = Image.fromarray(img).resize((416, 416)).convert('RGB')
    panel = Image.new('RGB', (416, 464), 'black')
    panel.paste(canvas, (0, 48))
    draw = ImageDraw.Draw(panel)
    label = (f'step={step:3d} act={ACTION_NAMES[action]} '
             f'rtg={RTG_TARGET:.1f} ret={total_reward:+.2f} wells={n_well_rewards} L={L}')
    if stagnation > 0: label += f' stuck={stagnation} T={temp:.1f}'
    draw.text((6, 6),  label,                 fill='white')
    draw.text((6, 24), f'pose=({x},{y},{d})', fill='white')
    frames.append(np.array(panel))

    _, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if reward > 0.5: n_well_rewards += 1
    oh = np.zeros(5, dtype=np.float32); oh[action] = 1.0
    hist_action[-1] = oh
    if term or trunc:
        break

print(f'rollout: {len(frames)} steps, total_reward={total_reward:+.3f}, '
      f'wells={n_well_rewards}')
imageio.mimwrite(MOVIE_PATH, frames, fps=10, codec='libx264')
print(f'wrote -> {MOVIE_PATH}')


## 6. Watch the movie


In [ ]:
from IPython.display import Video
Video(str(MOVIE_PATH), embed=False, width=500)
